In [ ]:
# ===========================================================================
# 0. 라이브러리 임포트 및 기본 설정
# ===========================================================================
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor # 사용되지 않았지만 유지
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import os
import shutil
from google.colab import drive
from collections import OrderedDict # CLI 입력 처리를 위해 추가
from xgboost import XGBRegressor

# Google Drive 마운트
DRIVE_MOUNT_PATH = '/content/drive'
drive.mount(DRIVE_MOUNT_PATH, force_remount=True)

# 경로 설정
BASE_PROJECT_PATH = '/content/drive/MyDrive' # 사용자의 기본 프로젝트 경로
KAGGLE_CONFIG_PATH_DRIVE = os.path.join(BASE_PROJECT_PATH, '.config/kaggle/kaggle.json') # Kaggle API 키 경로
DATASET_DOWNLOAD_PATH = os.path.join(BASE_PROJECT_PATH, 'ML_Project/sleep-efficiency') # 데이터셋 저장 경로
MODEL_SAVE_DIR = os.path.join(BASE_PROJECT_PATH, 'ML_Project/saved_models') # 학습된 모델 저장 경로

# 모델 학습 관련 설정값
RANDOM_STATE = 42
TEST_SIZE = 0.2
CV_FOLDS = 5

# 피처 및 타겟 컬럼 정의
TARGET_COLUMNS = ['Sleep efficiency', 'REM sleep percentage', 'Deep sleep percentage', 'Light sleep percentage']
BASE_SELECTED_FEATURES = [ # 전처리 함수에 전달할 '기본' 피처 목록 (모델 학습용)
    'Age', 'Gender', 'Sleep duration', 'Awakenings',
    'Caffeine consumption', 'Alcohol consumption', 'Smoking status', # Smoking status는 모델 입력용 (Yes/No)
    'Exercise frequency'
]

Mounted at /content/drive


In [ ]:
# ===========================================================================
# 1. 데이터 로드 및 준비 함수 (Kaggle API 사용)
# ===========================================================================
def setup_kaggle_api(drive_kaggle_json_path):
    """Kaggle API 키를 Colab 환경에 설정합니다."""
    kaggle_config_dir = os.path.expanduser('~/.kaggle')
    kaggle_json_path_colab = os.path.join(kaggle_config_dir, 'kaggle.json')
    os.makedirs(kaggle_config_dir, exist_ok=True)
    try:
        shutil.copyfile(drive_kaggle_json_path, kaggle_json_path_colab)
        os.chmod(kaggle_json_path_colab, 0o600)
        print("Kaggle API key configured successfully.")
    except Exception as e:
        print(f"Error configuring Kaggle API key: {e}")

def download_and_load_dataset(dataset_name, download_path, csv_filename):
    """Kaggle에서 데이터셋을 다운로드하고 CSV 파일을 DataFrame으로 로드합니다."""
    os.makedirs(download_path, exist_ok=True)
    try:
        import kaggle
        kaggle.api.dataset_download_files(dataset_name, path=download_path, unzip=True)
        print(f"Dataset '{dataset_name}' downloaded and unzipped to '{download_path}'.")
        df = pd.read_csv(os.path.join(download_path, csv_filename))
        print(f"'{csv_filename}' loaded successfully.")
        return df
    except Exception as e:
        print(f"Error in dataset download/load: {e}")
        return pd.DataFrame()

In [ ]:
# ===========================================================================
# 2. 전처리 파이프라인 함수 (모델 학습용 데이터셋 대상)
# ===========================================================================
def preprocess_data_pipeline(df, base_selected_features_for_model, target_columns, test_size=TEST_SIZE, random_state=RANDOM_STATE):
    """데이터를 전처리하여 모델 학습에 적합한 형태로 만듭니다."""
    print("\n--- Starting Data Preprocessing Pipeline ---")
    processed_df = df.copy()
    cols_to_drop_initial = ['ID', 'Bedtime', 'Wakeup time'] # 초기 불필요 컬럼 제거
    processed_df.drop(columns=[col for col in cols_to_drop_initial if col in processed_df.columns], inplace=True, errors='ignore')

    missing_val_cols = ['Awakenings', 'Caffeine consumption', 'Alcohol consumption', 'Exercise frequency']
    median_values = {} # 학습 데이터의 중앙값을 저장하여 추론 시 사용

    # 값 조정 (모델 입력용 클리핑)
    if 'Caffeine consumption' in processed_df.columns:
        processed_df['Caffeine consumption'] = processed_df['Caffeine consumption'].clip(upper=200, lower=0)
    if 'Alcohol consumption' in processed_df.columns: # 원본 데이터는 oz 단위로 가정하고 처리, 추론 시에는 mL -> oz 변환
        processed_df['Alcohol consumption'] = processed_df['Alcohol consumption'].clip(upper=5, lower=0) # 5oz 상한
    if 'Exercise frequency' in processed_df.columns:
        processed_df['Exercise frequency'] = processed_df['Exercise frequency'].clip(upper=5, lower=0) # 주 5회 상한


    # 인코딩 (Gender, Smoking status)
    if 'Gender' in base_selected_features_for_model and 'Gender' in processed_df.columns:
        processed_df = pd.get_dummies(processed_df, columns=['Gender'], drop_first=True, prefix='Gender')
        if 'Gender_Male' in processed_df.columns: processed_df['Gender_Male'] = processed_df['Gender_Male'].astype(float)
    if 'Smoking status' in base_selected_features_for_model and 'Smoking status' in processed_df.columns: # 모델은 Yes/No만 사용
        processed_df = pd.get_dummies(processed_df, columns=['Smoking status'], drop_first=True, prefix='Smoking_status') # Smoking_status_Yes 생성
        if 'Smoking_status_Yes' in processed_df.columns: processed_df['Smoking_status_Yes'] = processed_df['Smoking_status_Yes'].astype(float)

    for col in ['Age', 'Sleep duration']: # 숫자형 변환
        if col in base_selected_features_for_model and col in processed_df.columns: processed_df[col] = processed_df[col].astype(float)

    # 조건 기반 행 제거 (데이터 정제)
    initial_rows = len(processed_df)
    conditions = []
    if 'Age' in processed_df.columns: conditions.append(~((processed_df['Age'] < 18) | (processed_df['Age'] > 65)))
    if 'Sleep duration' in processed_df.columns: conditions.append(~((processed_df['Sleep duration'] < 4) | (processed_df['Sleep duration'] > 12))) # 범위 약간 조정
    if 'Sleep efficiency' in processed_df.columns: conditions.append(~(processed_df['Sleep efficiency'] < 0.5))
    if conditions:
        final_condition = pd.Series(True, index=processed_df.index)
        for cond in conditions: final_condition &= cond
        processed_df = processed_df[final_condition]
    print(f"Rows dropped by conditions: {initial_rows - len(processed_df)}")

    # 최종 피처셋 및 타겟 분리
    final_encoded_features_for_X = []
    for feature in base_selected_features_for_model: # BASE_SELECTED_FEATURES에 명시된 것만 사용
        if feature == 'Gender' and 'Gender_Male' in processed_df.columns: final_encoded_features_for_X.append('Gender_Male')
        elif feature == 'Smoking status' and 'Smoking_status_Yes' in processed_df.columns: final_encoded_features_for_X.append('Smoking_status_Yes')
        elif feature in processed_df.columns: final_encoded_features_for_X.append(feature)
    print(f"Final selected features for X (after encoding): {final_encoded_features_for_X}")

    X = processed_df[final_encoded_features_for_X]
    Y_dict = {target: processed_df[target] for target in target_columns if target in processed_df.columns}
    if not Y_dict or X.empty: return None, None, None, None, None, None, None

    # 학습/테스트 분할
    first_target_name = list(Y_dict.keys())[0]
    X_train, X_test, _, _ = train_test_split(X, Y_dict[first_target_name], test_size=test_size, random_state=random_state)
    y_train_dict = {target: Y_dict[target].loc[X_train.index] for target in Y_dict}
    y_test_dict = {target: Y_dict[target].loc[X_test.index] for target in Y_dict}

    # 분할 후 결측치 처리 (학습 데이터 중앙값 사용)
    for col in missing_val_cols:
        if col in X_train.columns:
            median_val = X_train[col].median()
            median_values[col] = median_val # 추론 시 사용하기 위해 저장
            X_train.loc[:, col] = X_train[col].fillna(median_val)
            if col in X_test.columns: X_test.loc[:, col] = X_test[col].fillna(median_val)

    # 이상치 제거 (Z-score 기반)
    outlier_cols = ['Age', 'Sleep duration', 'Awakenings', 'Caffeine consumption', 'Alcohol consumption', 'Exercise frequency']
    outlier_cols_present_train = [col for col in outlier_cols if col in X_train.columns]
    if outlier_cols_present_train:
        z_threshold = 3
        train_means = X_train[outlier_cols_present_train].mean()
        train_stds = X_train[outlier_cols_present_train].std()
        train_stds[train_stds == 0] = 1e-6 # 표준편차가 0인 경우 방지
        z_scores_train = ((X_train[outlier_cols_present_train] - train_means) / train_stds).abs()
        outlier_mask_train = (z_scores_train > z_threshold).any(axis=1)
        X_train = X_train[~outlier_mask_train]; y_train_dict = {k:v[~outlier_mask_train] for k,v in y_train_dict.items()}
        if not X_test.empty and outlier_cols_present_train: # 테스트셋에도 동일 기준 적용
            z_scores_test = ((X_test[outlier_cols_present_train] - train_means) / train_stds).abs()
            outlier_mask_test = (z_scores_test > z_threshold).any(axis=1)
            X_test = X_test[~outlier_mask_test]; y_test_dict = {k:v[~outlier_mask_test] for k,v in y_test_dict.items()}
        print(f"Rows removed by Z-score outlier detection from X_train: {sum(outlier_mask_train)}")

    # 스케일링 (StandardScaler)
    scalers = {} # 스케일러 저장하여 추론 시 사용
    cols_to_scale = ['Age', 'Sleep duration', 'Awakenings', 'Caffeine consumption', 'Alcohol consumption', 'Exercise frequency']
    cols_to_scale_present_train = [col for col in cols_to_scale if col in X_train.columns]
    X_train_scaled = X_train.copy(); X_test_scaled = X_test.copy()
    if cols_to_scale_present_train:
        for col in cols_to_scale_present_train:
            scaler = StandardScaler()
            X_train_scaled.loc[:, col] = scaler.fit_transform(X_train[[col]]).flatten()
            if not X_test.empty and col in X_test.columns:
                 X_test_scaled.loc[:, col] = scaler.transform(X_test[[col]]).flatten()
            scalers[col] = scaler
    print("--- Data Preprocessing Pipeline Completed ---")
    return X_train_scaled, X_test_scaled, y_train_dict, y_test_dict, scalers, median_values, final_encoded_features_for_X

In [ ]:
# ===========================================================================
# 3. 모델 학습 및 평가 함수 (GridSearchCV 사용)
# ===========================================================================
def train_and_evaluate_model_with_gridsearch(X_train, X_test, y_train, y_test, model_name, param_grid, cv=CV_FOLDS, random_state=RANDOM_STATE):
    """지정된 모델과 하이퍼파라미터로 GridSearchCV를 사용하여 학습하고 평가합니다."""
    print(f"\n--- Training and Evaluating {model_name} Regressor ---")
    if model_name == "RandomForest": base_model = RandomForestRegressor(random_state=random_state, n_jobs=-1)
    elif model_name == "SVR": base_model = SVR()
    elif model_name == "XGBoost": base_model = XGBRegressor(random_state=random_state, n_jobs=-1)
    else: raise ValueError(f"Unsupported model_name: {model_name}")

    if not param_grid: # 하이퍼파라미터 그리드가 없으면 기본값으로 학습
        print(f"No hyperparameter grid for {model_name}. Training with default parameters.")
        best_model = base_model; best_model.fit(X_train, y_train)
    else:
        grid_search = GridSearchCV(estimator=base_model, param_grid=param_grid, cv=cv, scoring='r2', verbose=1, n_jobs=-1) # 점수 기준 R2로 변경
        grid_search.fit(X_train, y_train)
        print(f"Best hyperparameters for {model_name}: {grid_search.best_params_}")
        best_model = grid_search.best_estimator_
    predictions = best_model.predict(X_test)
    mse = mean_squared_error(y_test, predictions); r2 = r2_score(y_test, predictions)
    print(f"Test Set Metrics for {model_name} -> MSE: {mse:.4f}, R2 Score: {r2:.4f}")
    return best_model, {"mse": mse, "r2": r2}, predictions

In [ ]:
# ===========================================================================
# 4. 학습 파이프라인 실행 함수
# ===========================================================================
def run_training_pipeline(raw_df, base_selected_features_for_model, target_columns, target_specific_model_configs, model_save_dir):
    """전체 학습 파이프라인을 실행하고, 학습된 모델과 전처리 도구들을 저장합니다."""
    if raw_df.empty: print("Raw DataFrame is empty. Training aborted."); return None, None, None, None

    # 전처리 실행
    X_train, X_test, y_train_all, y_test_all, learned_scalers, learned_medians, final_features_list = preprocess_data_pipeline(
        raw_df, base_selected_features_for_model, target_columns
    )
    if X_train is None or X_train.empty: print("Preprocessing failed or resulted in empty training set."); return None, None, None, None

    trained_models = {}; all_model_metrics = {}
    # 각 타겟 변수에 대해 모델 학습
    for target_name, y_train_target in y_train_all.items():
        print(f"\n<<<<< Processing Target for Training: {target_name} >>>>>")
        y_test_target = y_test_all[target_name]
        config = target_specific_model_configs.get(target_name)
        model_to_train = config.get('model_name', "RandomForest") if config else "RandomForest"
        params_to_use = config.get('param_grid', {}) if config else {}
        best_model, metrics, _ = train_and_evaluate_model_with_gridsearch(
            X_train, X_test, y_train_target, y_test_target, model_name=model_to_train, param_grid=params_to_use
        )
        model_dict_key = f"{model_to_train}_{target_name.replace(' ', '_').lower()}" # 키 이름 소문자 및 일관성있게 변경
        trained_models[model_dict_key] = best_model; all_model_metrics[model_dict_key] = metrics

    print("\n\n===== Overall Model Performance Summary (Training Pipeline) =====")
    for k, v in all_model_metrics.items(): print(f"Model: {k} -> MSE: {v['mse']:.4f}, R2: {v['r2']:.4f}")

    # 학습된 모델 및 전처리기 저장
    os.makedirs(model_save_dir, exist_ok=True)
    for k, m in trained_models.items(): joblib.dump(m, os.path.join(model_save_dir, f"best_{k}_model.pkl")) # 파일명에 target_name 포함
    joblib.dump(learned_scalers, os.path.join(model_save_dir, 'scalers.pkl'))
    joblib.dump(learned_medians, os.path.join(model_save_dir, 'medians.pkl'))
    joblib.dump(final_features_list, os.path.join(model_save_dir, 'final_features_list.pkl')) # 학습에 사용된 최종 피처 목록 저장
    joblib.dump(base_selected_features_for_model, os.path.join(model_save_dir, 'base_selected_features_for_model.pkl')) # 기본 피처 목록도 저장
    print(f"Trained models, scalers, median values, and feature lists saved to '{model_save_dir}'.")
    return trained_models, learned_scalers, learned_medians, final_features_list

In [ ]:
# ===========================================================================
# 5. 추론(사용자 입력 처리 및 예측) 함수
# ===========================================================================
def transform_user_input_for_prediction(user_raw_input_dict, final_feature_list_from_training, scalers_from_training, medians_from_training):
    """사용자 입력을 모델 예측에 적합한 형태로 변환합니다."""
    print("\n--- Transforming User Input for Prediction ---")
    user_df_for_model = pd.DataFrame([user_raw_input_dict.copy()]) # 딕셔너리를 DataFrame으로

    # 'Awakenings': "잘 모르겠다" 또는 숫자 처리
    awakening_input = str(user_df_for_model.get('Awakenings', medians_from_training.get('Awakenings', 0)).iloc[0])
    processed_awakening_value = medians_from_training.get('Awakenings', 2) # 기본값: 중앙값 또는 2
    if awakening_input.lower() == "잘 모르겠다":
        pass # 중앙값 사용
    else:
        numeric_awakening = pd.to_numeric(awakening_input.replace("회", "").replace(" 이상", "").strip(), errors='coerce')
        if pd.notna(numeric_awakening): processed_awakening_value = numeric_awakening
    user_df_for_model.loc[0, 'Awakenings'] = np.clip(processed_awakening_value, 0, 4) # 0~4회 범위로 클립

    # 'Alcohol consumption': mL 입력 -> oz 변환 및 clip (모델 학습 시 oz 단위 사용 가정)
    alcohol_input_ml = user_df_for_model.get('Alcohol consumption', "0").iloc[0]
    alcohol_ml_numeric = pd.to_numeric(alcohol_input_ml, errors='coerce')
    if pd.notna(alcohol_ml_numeric):
        user_df_for_model.loc[0, 'Alcohol consumption'] = np.clip(alcohol_ml_numeric / 29.5735, 0, 5) # mL을 oz로 변환 (1oz ~ 29.57mL), 5oz 상한
    else: # 변환 실패 시 중앙값(oz 단위) 사용
        user_df_for_model.loc[0, 'Alcohol consumption'] = medians_from_training.get('Alcohol consumption', 0)

    # 'Exercise frequency': 직접 숫자 입력 -> clip (모델 학습 시 0~5회 범위 사용 가정)
    exercise_input = user_df_for_model.get('Exercise frequency', "0").iloc[0]
    exercise_freq_numeric = pd.to_numeric(exercise_input, errors='coerce')
    if pd.notna(exercise_freq_numeric):
        user_df_for_model.loc[0, 'Exercise frequency'] = np.clip(exercise_freq_numeric, 0, 5) # 0~5회 범위로 클립
    else: # 변환 실패 시 중앙값 사용
        user_df_for_model.loc[0, 'Exercise frequency'] = medians_from_training.get('Exercise frequency', 0)

    # 'Caffeine consumption': mg 입력 -> clip (모델 학습 시 0~200mg 범위 사용 가정)
    caffeine_input_mg = user_df_for_model.get('Caffeine consumption', "0").iloc[0]
    caffeine_mg_numeric = pd.to_numeric(caffeine_input_mg, errors='coerce')
    if pd.notna(caffeine_mg_numeric):
        user_df_for_model.loc[0, 'Caffeine consumption'] = np.clip(caffeine_mg_numeric, 0, 200) # 0~200mg 범위로 클립
    else: # 변환 실패 시 중앙값 사용
        user_df_for_model.loc[0, 'Caffeine consumption'] = medians_from_training.get('Caffeine consumption', 0)


    # Age, Sleep duration: 숫자형으로 변환
    for col in ['Age', 'Sleep duration']:
        input_val = user_df_for_model.get(col, medians_from_training.get(col, 0)).iloc[0]
        numeric_val = pd.to_numeric(input_val, errors='coerce')
        user_df_for_model.loc[0, col] = numeric_val if pd.notna(numeric_val) else medians_from_training.get(col, 0)

    # 모델 입력에 필요한 숫자형 컬럼들 float으로 변환
    model_input_numeric_cols = ['Age', 'Sleep duration', 'Awakenings', 'Caffeine consumption', 'Alcohol consumption', 'Exercise frequency']
    for col in model_input_numeric_cols:
        if col in user_df_for_model.columns:
            if pd.isna(pd.to_numeric(user_df_for_model[col].iloc[0], errors='coerce')): # 숫자로 변환 안되면 중앙값
                 user_df_for_model.loc[0, col] = medians_from_training.get(col, 0)
            user_df_for_model[col] = user_df_for_model[col].astype(float)

    # Gender 인코딩: 'Male' -> 1.0, 'Female' -> 0.0
    if 'Gender' in user_raw_input_dict:
        user_df_for_model['Gender_Male'] = 1.0 if user_raw_input_dict['Gender'].lower() == 'male' else 0.0
    if 'Gender' in user_df_for_model.columns: user_df_for_model.drop(columns=['Gender'], inplace=True, errors='ignore')

    # Smoking status 인코딩 (모델용): 'Yes' -> 1.0, 'No' -> 0.0
    if 'Smoking status' in user_raw_input_dict: # 점수 계산용 Smoking_history_status와 별개
        user_df_for_model['Smoking_status_Yes'] = 1.0 if user_raw_input_dict['Smoking status'].lower() == 'yes' else 0.0
    if 'Smoking status' in user_df_for_model.columns: user_df_for_model.drop(columns=['Smoking status'], inplace=True, errors='ignore')


    # 학습 시 사용된 최종 피처 목록에 맞춰 컬럼 정리
    for col in final_feature_list_from_training:
        if col not in user_df_for_model.columns: user_df_for_model[col] = 0.0 # 누락된 피처는 0으로 채움 (주로 더미 변수)
    user_df_encoded = user_df_for_model[final_feature_list_from_training]

    # 스케일링 적용
    user_df_scaled = user_df_encoded.copy()
    for col, scaler in scalers_from_training.items(): # 저장된 스케일러 사용
        if col in user_df_scaled.columns:
            user_df_scaled.loc[:, col] = scaler.transform(user_df_scaled[[col]]).flatten()
    print("--- User Input Transformation Completed ---")
    return user_df_scaled


def run_inference_pipeline(user_raw_input_dict, target_columns, target_specific_model_configs, model_save_dir):
    """저장된 모델과 전처리 도구를 로드하여 사용자 입력에 대한 추론을 실행합니다."""
    print("\n--- Running Inference Pipeline ---")
    try:
        loaded_scalers = joblib.load(os.path.join(model_save_dir, 'scalers.pkl'))
        loaded_medians = joblib.load(os.path.join(model_save_dir, 'medians.pkl'))
        final_features_list = joblib.load(os.path.join(model_save_dir, 'final_features_list.pkl'))
    except FileNotFoundError:
        print("Error: Preprocessing tools (scalers, medians, feature_list) not found. Run training first."); return None, user_raw_input_dict

    user_input_processed_for_model = transform_user_input_for_prediction(
        user_raw_input_dict, final_features_list, loaded_scalers, loaded_medians
    )
    if user_input_processed_for_model is None or user_input_processed_for_model.empty:
        print("User input processing failed."); return None, user_raw_input_dict

    user_model_predictions = {}
    print("\n--- User Model Predictions ---")
    for target_name in target_columns: # 모든 타겟에 대해 예측
        model_type = target_specific_model_configs.get(target_name, {}).get('model_name', 'RandomForest')
        target_name_key = target_name.replace(' ', '_').lower()
        model_filename = os.path.join(model_save_dir, f"best_{model_type}_{target_name_key}_model.pkl")
        try:
            loaded_model = joblib.load(model_filename)
            prediction = loaded_model.predict(user_input_processed_for_model)
            # 예측값 범위 보정 (필요시)
            if target_name == 'Sleep efficiency': prediction[0] = np.clip(prediction[0], 0.5, 1.0)
            elif 'percentage' in target_name: prediction[0] = np.clip(prediction[0], 0, 100)

            user_model_predictions[target_name] = float(prediction[0])
            print(f"Predicted {target_name}: {user_model_predictions[target_name]:.2f}")
        except FileNotFoundError:
            user_model_predictions[target_name] = None # 예측 실패 시 None
            print(f"Error: Model for {target_name} not found at {model_filename}")
    return user_model_predictions, user_raw_input_dict # 점수계산을 위해 원본 입력도 반환

In [ ]:
# ===========================================================================
# 6. 수면 점수 계산 함수
# ===========================================================================
def calculate_sleep_score(user_raw_input_for_score, model_predictions):
    """사용자 입력과 모델 예측값을 바탕으로 수면 점수를 계산합니다."""
    total_score = 0
    detailed_scores = {} # 항목별 점수 저장

    # 점수 계산에 필요한 값 추출 및 형 변환
    age = pd.to_numeric(user_raw_input_for_score.get('Age'), errors='coerce')
    sleep_duration = pd.to_numeric(user_raw_input_for_score.get('Sleep duration'), errors='coerce')

    awakenings_input = str(user_raw_input_for_score.get('Awakenings', '2')) # 기본값 2회
    if awakenings_input.lower() == "잘 모르겠다": awakenings_for_score = 2 # "잘 모르겠다"는 점수 계산 시 2회로 간주
    else:
        awakenings_for_score = pd.to_numeric(awakenings_input.replace("회","").replace(" 이상","").strip(), errors='coerce')
        if pd.isna(awakenings_for_score): awakenings_for_score = 2 # 변환 실패시 2회

    # --- 1. 핵심 수면 구조 (총 53점) ---
    score_cat1 = 0
    # 1.1 평균 수면 시간 (10점)
    s1_1 = 0
    if pd.notna(age) and pd.notna(sleep_duration) and 18 <= age <= 64:
        if sleep_duration < 6: s1_1 = 0
        elif 6 <= sleep_duration < 7: s1_1 = 8
        elif 7 <= sleep_duration < 9: s1_1 = 10
        elif 9 <= sleep_duration < 10: s1_1 = 5
        else: s1_1 = 3 # 10시간 이상
    detailed_scores['1.1_Avg_Sleep_Time'] = s1_1; score_cat1 += s1_1

    # 1.2 평균 각성 횟수 (10점)
    s1_2 = 0
    if pd.notna(awakenings_for_score):
        if awakenings_for_score <= 1: s1_2 = 10
        elif awakenings_for_score == 2: s1_2 = 6
        elif awakenings_for_score == 3: s1_2 = 2
        else: s1_2 = 0 # 4회 이상
    detailed_scores['1.2_Awakenings_Count'] = s1_2; score_cat1 += s1_2

    # 1.3.1 수면 효율성 (13점) - 모델 예측값 사용
    s1_3_1 = 0
    se_pred_percent = model_predictions.get('Sleep efficiency', 0) * 100 if model_predictions.get('Sleep efficiency') else 0
    if se_pred_percent >= 90: s1_3_1 = 13
    elif 85 <= se_pred_percent < 90: s1_3_1 = 10
    elif 80 <= se_pred_percent < 85: s1_3_1 = 8
    elif 75 <= se_pred_percent < 80: s1_3_1 = 5
    else: s1_3_1 = 3 # 75% 미만
    detailed_scores['1.3.1_Sleep_Efficiency'] = s1_3_1; score_cat1 += s1_3_1

    # 1.3.2 깊은 수면 비율 (10점) - 모델 예측값 사용
    s1_3_2 = 0
    deep_pred_percent = model_predictions.get('Deep sleep percentage', 0) if model_predictions.get('Deep sleep percentage') else 0
    if 20 <= deep_pred_percent <= 25: s1_3_2 = 10
    elif (15 <= deep_pred_percent < 20) or (25 < deep_pred_percent <= 30): s1_3_2 = 8
    elif 10 <= deep_pred_percent < 15: s1_3_2 = 5
    else: s1_3_2 = 0 # 10% 미만 또는 30% 초과
    detailed_scores['1.3.2_Deep_Sleep_Ratio'] = s1_3_2; score_cat1 += s1_3_2

    # 1.3.3 얕은 수면 비율 (5점) - 모델 예측값 사용
    s1_3_3 = 0
    light_pred_percent = model_predictions.get('Light sleep percentage', 0) if model_predictions.get('Light sleep percentage') else 0
    if 50 <= light_pred_percent <= 60: s1_3_3 = 5
    elif (30 <= light_pred_percent < 50) or (60 < light_pred_percent <= 70): s1_3_3 = 3
    else: s1_3_3 = 0 # 30% 미만 또는 70% 초과
    detailed_scores['1.3.3_Light_Sleep_Ratio'] = s1_3_3; score_cat1 += s1_3_3

    # 1.3.4 REM 수면 비율 (5점) - 모델 예측값 사용
    s1_3_4 = 0
    rem_pred_percent = model_predictions.get('REM sleep percentage', 0) if model_predictions.get('REM sleep percentage') else 0
    if 20 <= rem_pred_percent <= 25: s1_3_4 = 5
    elif (15 <= rem_pred_percent < 20) or (25 < rem_pred_percent <= 30): s1_3_4 = 3
    elif 10 <= rem_pred_percent < 15: s1_3_4 = 1
    else: s1_3_4 = 0 # 10% 미만 또는 15% 미만이면서 특정 조건 아닐 때 (점수표 기준에 따라 0점 구간 명확히)
    detailed_scores['1.3.4_REM_Sleep_Ratio'] = s1_3_4; score_cat1 += s1_3_4
    detailed_scores['Category1_Core_Structure_Total(Max53)'] = score_cat1
    total_score += score_cat1

    # --- 2. 수면 시간 및 규칙성 (총 27점) ---
    score_cat2 = 0
    bed_std = pd.to_numeric(user_raw_input_for_score.get('Bedtime_std_dev'), errors='coerce')
    wake_std = pd.to_numeric(user_raw_input_for_score.get('Wakeup_time_std_dev'), errors='coerce')
    wk_wknd_diff_hours = pd.to_numeric(user_raw_input_for_score.get('Weekday_weekend_sleep_diff'), errors='coerce') # 시간 단위
    bed_consistency_days_input = user_raw_input_for_score.get('Bedtime_consistency_days') # 숫자 또는 문자열
    # Bedtime_consistency_days: 숫자면 그대로, 문자열이면 숫자 변환 시도 (map_survey_to_notebook_input에서 이미 숫자로 변환됨)
    bed_consistency_days = pd.to_numeric(bed_consistency_days_input, errors='coerce')
    if pd.isna(bed_consistency_days) and isinstance(bed_consistency_days_input, str): # 예: "7일 모두" -> 7 (이런 변환은 map에서 처리)
        consistency_map_reverse = {'7일 모두': 7, '5~6일': 5.5, '3~4일': 3.5, '3일 미만': 1, '매우 일관됨':7, '상당히 일관됨':5.5, '다소 불일관됨':3.5, '매우 불일관됨':1}
        bed_consistency_days = consistency_map_reverse.get(bed_consistency_days_input, 0)

    bed_slot_str = user_raw_input_for_score.get('Bedtime_slot')
    s2_1_1, s2_1_2, s2_1_3, s2_2_1, s2_2_2 = 0,0,0,0,0 # 각 항목 점수 초기화

    if pd.notna(bed_std): # 취침 시간 표준편차
        if bed_std <= 30: s2_1_1 = 6
        elif bed_std <= 60: s2_1_1 = 4
        elif bed_std <= 90: s2_1_1 = 2
        else: s2_1_1 = 1
    if pd.notna(wake_std): # 기상 시간 표준편차
        if wake_std <= 30: s2_1_2 = 6
        elif wake_std <= 60: s2_1_2 = 4
        elif wake_std <= 90: s2_1_2 = 2
        else: s2_1_2 = 1
    if pd.notna(wk_wknd_diff_hours): # 주중-주말 수면 시간 차이
        if wk_wknd_diff_hours < 1: s2_1_3 = 5
        elif wk_wknd_diff_hours <= 1.5: s2_1_3 = 3
        elif wk_wknd_diff_hours <= 2: s2_1_3 = 1
        else: s2_1_3 = 0
    if pd.notna(bed_consistency_days): # 취침 시간대 일관성
        if bed_consistency_days >= 6.5: s2_2_1 = 5 # 7일 (매우 일관)
        elif bed_consistency_days >= 4.5: s2_2_1 = 3 # 5~6일 (상당히 일관)
        elif bed_consistency_days >= 2.5: s2_2_1 = 1 # 3~4일 (다소 불일관)
        else: s2_2_1 = 0 # 3일 미만 (매우 불일관)
    if bed_slot_str: # 늦은 취침 시간대
        if bed_slot_str == '새벽 1시 이전 취침': s2_2_2 = 5
        elif bed_slot_str == '새벽 1-2시 사이 취침': s2_2_2 = 3
        elif bed_slot_str == '새벽 2-3시 사이 취침': s2_2_2 = 1
        else: s2_2_2 = 0 # 새벽 3시 이후

    detailed_scores['2.1.1_Bedtime_StdDev'] = s2_1_1; score_cat2 += s2_1_1
    detailed_scores['2.1.2_Wakeup_StdDev'] = s2_1_2; score_cat2 += s2_1_2
    detailed_scores['2.1.3_Weekday_Weekend_Diff'] = s2_1_3; score_cat2 += s2_1_3
    detailed_scores['2.2.1_Bedtime_Consistency'] = s2_2_1; score_cat2 += s2_2_1
    detailed_scores['2.2.2_Bedtime_Slot'] = s2_2_2; score_cat2 += s2_2_2
    detailed_scores['Category2_Regularity_Total(Max27)'] = score_cat2
    total_score += score_cat2

    # --- 3. 생활 습관 요인 (총 20점) ---
    score_cat3 = 0
    # 3.1 카페인 (5점)
    caffeine_mg_raw_score = pd.to_numeric(user_raw_input_for_score.get('Caffeine consumption'), errors='coerce') # 원본 mg값 사용
    caffeine_last_h = pd.to_numeric(user_raw_input_for_score.get('Caffeine_last_consumed_hours_before_bed'), errors='coerce')
    s3_1 = 0
    if pd.notna(caffeine_mg_raw_score) and (pd.notna(caffeine_last_h) or caffeine_mg_raw_score == 0): # 0mg이면 시간 무관
        if caffeine_mg_raw_score == 0: s3_1 = 5
        elif caffeine_mg_raw_score < 100:
            if caffeine_last_h > 8: s3_1 = 4
            elif 4 <= caffeine_last_h <= 8: s3_1 = 3
            else: s3_1 = 1
        elif 100 <= caffeine_mg_raw_score <= 200:
            if caffeine_last_h > 10: s3_1 = 3
            elif 6 <= caffeine_last_h <= 10: s3_1 = 2
            else: s3_1 = 0
        elif 200 < caffeine_mg_raw_score <= 400:
            if caffeine_last_h > 12: s3_1 = 2
            elif 8 <= caffeine_last_h <= 12: s3_1 = 1
            else: s3_1 = 0
        else: s3_1 = 0 # 400mg 초과
    detailed_scores['3.1_Caffeine'] = s3_1; score_cat3 += s3_1

    # 3.2 알코올 (5점)
    alcohol_ml_raw = pd.to_numeric(user_raw_input_for_score.get('Alcohol consumption'), errors='coerce') # 원본 mL값 사용
    alcohol_drinks_calc = 0 # "잔" 수로 변환 (1잔 = 순수 알코올 약 10g)
    if pd.notna(alcohol_ml_raw): # 주종을 모르므로, 대략적인 가이드라인 (소주 1잔 50ml 17% = 8.5g, 맥주 1잔 200ml 5% = 10g)
        pure_alcohol_g = alcohol_ml_raw * 0.1 # 거칠게 10% 알코올로 가정하여 g 계산 (개선 필요)
        if pure_alcohol_g == 0 : alcohol_drinks_calc = 0
        elif pure_alcohol_g <= 12: alcohol_drinks_calc = 1 # 약 1잔
        elif pure_alcohol_g <= 24: alcohol_drinks_calc = 2 # 약 2잔
        else: alcohol_drinks_calc = 3 # 3잔 이상
    alcohol_last_h = pd.to_numeric(user_raw_input_for_score.get('Alcohol_last_consumed_hours_before_bed'), errors='coerce')
    s3_2 = 0
    if pd.notna(alcohol_drinks_calc) and (pd.notna(alcohol_last_h) or alcohol_drinks_calc == 0):
        if alcohol_drinks_calc == 0: s3_2 = 5
        elif alcohol_drinks_calc == 1:
            if alcohol_last_h > 4: s3_2 = 3
            elif 2 <= alcohol_last_h <= 4: s3_2 = 2
            else: s3_2 = 1
        elif alcohol_drinks_calc == 2:
            if alcohol_last_h > 6: s3_2 = 2
            elif 3 <= alcohol_last_h <= 6: s3_2 = 1
            else: s3_2 = 0
        else: s3_2 = 0 # 3잔 이상
    detailed_scores['3.2_Alcohol'] = s3_2; score_cat3 += s3_2

    # 3.3 흡연 (5점)
    smoking_history_for_score = user_raw_input_for_score.get('Smoking_history_status', "비흡연자")
    s3_3 = 0
    if smoking_history_for_score == "비흡연자": s3_3 = 5
    elif smoking_history_for_score == "과거흡연_1년이상": s3_3 = 3
    elif smoking_history_for_score == "현재흡연_경증" or smoking_history_for_score == "과거흡연_1년미만": s3_3 = 1 # 갑년<10 또는 10개비 미만
    else: s3_3 = 0 # 현재흡연_중증 (갑년>=10 또는 10개비 이상)
    detailed_scores['3.3_Smoking'] = s3_3; score_cat3 += s3_3

    # 3.4 운동 (5점)
    s3_4 = 0
    moderate_mins = pd.to_numeric(user_raw_input_for_score.get('Exercise_aerobic_moderate_minutes'), errors='coerce')
    vigorous_mins = pd.to_numeric(user_raw_input_for_score.get('Exercise_aerobic_vigorous_minutes'), errors='coerce')
    exercise_last_h_str = user_raw_input_for_score.get('Exercise_last_hours_before_bed', "해당 없음")
    exercise_last_h_numeric = None
    if exercise_last_h_str != "해당 없음":
        exercise_last_h_numeric = pd.to_numeric(exercise_last_h_str, errors='coerce')

    # 운동량 및 타이밍에 따른 점수 계산 (점수표 기준)
    exercised_enough = (pd.notna(moderate_mins) and moderate_mins >= 150) or \
                       (pd.notna(vigorous_mins) and vigorous_mins >= 75)
    exercised_moderately = (pd.notna(moderate_mins) and 75 <= moderate_mins < 150) or \
                           (pd.notna(vigorous_mins) and 35 <= vigorous_mins < 75)
    exercised_lightly = (pd.notna(moderate_mins) and 30 <= moderate_mins < 75) or \
                        (pd.notna(vigorous_mins) and 15 <= vigorous_mins < 35)
    no_exercise = not (exercised_enough or exercised_moderately or exercised_lightly)
    if pd.notna(moderate_mins) and moderate_mins == 0 and pd.notna(vigorous_mins) and vigorous_mins == 0:
        no_exercise = True


    timing_good = pd.notna(exercise_last_h_numeric) and exercise_last_h_numeric > 3
    timing_bad = (pd.notna(exercise_last_h_numeric) and exercise_last_h_numeric <= 3) or \
                 (exercise_last_h_str != "해당 없음" and pd.isna(exercise_last_h_numeric)) # 시간이 입력됐는데 숫자가 아니면 안좋게

    if no_exercise or exercise_last_h_str == "해당 없음" and not (exercised_enough or exercised_moderately or exercised_lightly) : # 운동 안했거나, 정보 부족하면 0점
        s3_4 = 0
    elif exercised_enough: s3_4 = 5 if timing_good else 4
    elif exercised_moderately: s3_4 = 3 if timing_good else 2
    elif exercised_lightly: s3_4 = 2 if timing_good else 1
    else: s3_4 = 0 # 위의 어떤 조건에도 해당하지 않으면 0점 (예: 운동량은 있으나 시간 정보가 아예 없는 경우 등)

    detailed_scores['3.4_Exercise'] = s3_4; score_cat3 += s3_4 # 키 이름 통일
    detailed_scores['Category3_Lifestyle_Total(Max20)'] = score_cat3
    total_score += score_cat3

    return int(np.clip(total_score, 0, 100)), detailed_scores

In [ ]:
# ===========================================================================
# 7. 점수 해석 함수
# ===========================================================================
def get_score_interpretation(total_score):
    """총점에 따른 질적 평가와 일반적 시사점을 반환합니다."""
    if 90 <= total_score <= 100: return "최우수", "전반적으로 매우 건강한 수면 패턴을 유지하고 있습니다. 현재의 좋은 습관을 지속할 것을 권장합니다."
    elif 80 <= total_score <= 89: return "우수", "대부분의 수면 요소가 양호한 상태입니다. 일부 개선 가능한 영역이 있을 수 있으나 전반적으로 건강한 편입니다."
    elif 70 <= total_score <= 79: return "보통", "수면의 질에 약간의 개선이 필요한 부분이 있습니다. 점수가 낮은 항목을 확인하고 생활 습관 개선을 고려하세요."
    elif 60 <= total_score <= 69: return "개선 필요", "수면의 질에 몇 가지 우려되는 점이 있으며, 적극적인 개선 노력이 필요합니다. 전문가 상담도 고려해보세요."
    else: return "불량", "수면의 질이 상당히 낮아 건강에 부정적인 영향을 미칠 수 있습니다. 생활 습관 전반의 점검 및 전문가의 도움이 시급합니다."

In [ ]:
# ===========================================================================
# 8. 세부 피드백 생성 함수 (출력 형식 수정 및 로직 통합)
# ===========================================================================
def generate_detailed_feedback(detailed_scores, user_input_for_feedback, model_predictions_for_feedback):
    """항목별 점수, 사용자 입력, 모델 예측을 바탕으로 원인 분석을 포함한 세부 개선 조언을 생성합니다."""
    feedback_list = []
    age = pd.to_numeric(user_input_for_feedback.get('Age'), errors='coerce')

    print(f"\n######################### 사용자님의 수면 습관 피드백입니다. ############################\n")

    # =================================================================================
    # 1. 핵심 수면 구조 피드백
    # =================================================================================
    # 1.1 평균 수면 시간
    score_s1_1 = detailed_scores.get('1.1_Avg_Sleep_Time', 0)
    sleep_duration_val = pd.to_numeric(user_input_for_feedback.get('Sleep duration'), errors='coerce')
    if pd.notna(sleep_duration_val) and pd.notna(age) and 18 <= age <= 64:
        feedback_text = ""
        if score_s1_1 == 10:
            feedback_text = f"1. 하루 평균 수면 시간: 훌륭해요! 현재 사용자님의 하루 평균 수면 시간이 {sleep_duration_val:.1f}시간으로 권장 수면 시간에 맞춰서 잘 자고 계세요. 앞으로도 지금과 같은 수면 시간을 쭉 유지해봐요!"
        elif score_s1_1 == 8:
            feedback_text = f"1. 하루 평균 수면 시간: 현재 사용자님의 하루 평균 수면 시간이 {sleep_duration_val:.1f}시간으로 권장 수면 시간보다 약간 적어요. 가능하다면 하루 평균 수면 시간이 7~9시간이 되도록 자는 것을 추천드려요."
        elif score_s1_1 == 5 or score_s1_1 == 3:
            feedback_text = f"1. 하루 평균 수면 시간: 현재 사용자님의 하루 평균 수면 시간이 {sleep_duration_val:.1f}시간으로 권장 수면 시간보다 많아요. 과도한 수면은 건강에 좋지 않은 영향을 끼칠 수 있으므로 되도록 권장 수면 시간에 맞춰 수면을 취해주세요."
        else:  # 0점
            feedback_text = f"1. 하루 평균 수면 시간: 현재 사용자님의 하루 평균 수면 시간이 {sleep_duration_val:.1f}시간으로 너무 적어요. 하루 평균 7~9시간은 잘 수 있도록 해주셔야 해요! 자기 전 핸드폰 사용은 블루라이트 노출로 멜라토닌 분비를 억제해 수면 진입을 방해할 수 있습니다."
        score_text = f"[점수] : {score_s1_1}/10"
        feedback_list.append(f"{feedback_text}\n{score_text}")

    # 1.2 평균 각성 횟수
    score_s1_2 = detailed_scores.get('1.2_Awakenings_Count', 0)
    awakenings_str = str(user_input_for_feedback.get('Awakenings', '정보 없음'))
    awakenings_feedback_val = awakenings_str
    if awakenings_str.lower() == "잘 모르겠다": awakenings_feedback_val = "평균 2회"
    elif "회" not in awakenings_str: awakenings_feedback_val = f"평균 {awakenings_str}회"

    feedback_text = ""
    if score_s1_2 == 10:
        feedback_text = f"2. 수면 중 평균 각성 횟수: 아주 우수해요! 수면 중 {awakenings_feedback_val}로 거의 깨지 않아 수면 연속성이 매우 좋습니다. 현재 습관을 계속 유지해주세요."
    elif score_s1_2 == 6:
        feedback_text = f"2. 수면 중 평균 각성 횟수: 양호하지만 개선의 여지가 있습니다. 수면 중 {awakenings_feedback_val}로, 더 매끄러운 수면을 위해 잠들기 전 블루라이트 노출을 줄이고 규칙적인 수면 시간을 유지해보세요."
    else:  # 2점 or 0점
        feedback_text = f"2. 수면 중 평균 각성 횟수: 주의가 필요합니다. 수면 중 {awakenings_feedback_val}로 수면이 상당히 방해받고 있습니다. 이는 깊은 수면을 줄여 낮 시간 피로를 유발할 수 있습니다. 취침 전 카페인/알코올 섭취를 피하고, 어둡고 조용한 환경을 조성하는 것이 중요합니다."
    score_text = f"[점수] : {score_s1_2}/10"
    feedback_list.append(f"{feedback_text}\n{score_text}")

    # 1.3.1 수면 효율성
    score_s1_3_1 = detailed_scores.get('1.3.1_Sleep_Efficiency', 0)
    se_pred_val = model_predictions_for_feedback.get('Sleep efficiency', 0) * 100
    feedback_text = ""
    if score_s1_3_1 == 13:
        feedback_text = f"3. 수면 효율성: 훌륭해요! 예측된 수면 효율성은 {se_pred_val:.1f}%로, 침대에 누운 시간 대부분을 실제 수면에 사용하고 계십니다."
    elif score_s1_3_1 >= 8:  # 10, 8점
        feedback_text = f"3. 수면 효율성: 양호한 편입니다. 예측된 수면 효율성은 {se_pred_val:.1f}%로, 개선의 여지는 있지만 비교적 잠을 잘 이루는 편입니다."
    else:  # 5, 3점
        feedback_text = f"3. 수면 효율성: 개선이 필요합니다. 예측된 수면 효율성은 {se_pred_val:.1f}%로, 잠자리에 누워 깨어 있는 시간이 깁니다. 이는 입면이 어렵거나 중간에 자주 깨는 것을 의미하며, 만성 피로의 원인이 될 수 있습니다."
    score_text = f"[점수] : {score_s1_3_1}/13"
    feedback_list.append(f"{feedback_text}\n{score_text}")

    # 1.3.2 깊은 수면 비율
    score_s1_3_2 = detailed_scores.get('1.3.2_Deep_Sleep_Ratio', 0)
    deep_pred_val = model_predictions_for_feedback.get('Deep sleep percentage', 0)
    feedback_text = ""
    analysis_text = ""
    if score_s1_3_2 == 10:
        feedback_text = f"4. 깊은 수면 비율: 좋습니다! 예측된 깊은 수면 비율({deep_pred_val:.1f}%)이 신체 회복에 이상적인 범위입니다."
    else:
        feedback_text = f"4. 깊은 수면 비율: 예측된 깊은 수면 비율은 {deep_pred_val:.1f}%로, 이상적인 비율(20-25%)과 차이가 있습니다. 깊은 수면은 신체 회복과 면역 시스템 강화에 매우 중요합니다."
        causal_feedback_parts = []
        if detailed_scores.get('3.1_Caffeine', 5) <= 1: causal_feedback_parts.append("▶ 자기 전 카페인 섭취는 뇌에서의 피로 신호 물질(아데노신)을 차단시켜, 수면 압력이 충분히 쌓이지 않도록 방해하여 깊은 수면의 비율을 줄입니다. 되도록 취침 6시간 이내로는 카페인 섭취를 자제해주세요.")
        if detailed_scores.get('3.2_Alcohol', 5) <= 1: causal_feedback_parts.append("▶ 자기 전 알코올 섭취는 초반에는 깊은 수면을 증가시킬 수 있지만, 수면 중·후반부에서는 잦은 각성을 유도하여 실제 깊은 수면 시간을 크게 줄입니다. 되도록 취침 4시간 전 알코올 섭취는 자제해주세요.")
        if detailed_scores.get('3.3_Smoking', 5) <= 1: causal_feedback_parts.append("▶ 니코틴은 각성 효과를 유발하여 수면 중 각성과 파편화를 증가시켜 깊은 수면 단계로 안정적으로 진입하는 것을 방해합니다. 금연은 수면의 질을 높이는 데 큰 도움이 됩니다.")
        if causal_feedback_parts:
            analysis_text = "\n   [원인 분석]: 아래와 같은 생활 습관이 깊은 수면을 방해했을 수 있습니다.\n" + "\n".join([f"   {part}" for part in causal_feedback_parts])
    score_text = f"[점수] : {score_s1_3_2}/10"
    feedback_list.append(f"{feedback_text}{analysis_text}\n{score_text}")

    # 1.3.3 얕은 수면 비율
    score_s1_3_3 = detailed_scores.get('1.3.3_Light_Sleep_Ratio', 0)
    light_pred_val = model_predictions_for_feedback.get('Light sleep percentage', 0)
    feedback_text = ""
    analysis_text = ""
    if score_s1_3_3 == 5:
        feedback_text = f"5. 얕은 수면 비율: 좋습니다! 예측된 얕은 수면 비율({light_pred_val:.1f}%)이 적절한 범위입니다."
    else:
        causal_feedback_parts = []
        if light_pred_val > 60:
            feedback_text = f"5. 얕은 수면 비율: 예측된 얕은 수면 비율이 {light_pred_val:.1f}%로, 이상적인 비율(50-60%)보다 높습니다. 이는 숙면을 취하지 못하고 있다는 신호일 수 있습니다."
            if detailed_scores.get('3.1_Caffeine', 5) <= 1 or detailed_scores.get('3.2_Alcohol', 5) <= 1: causal_feedback_parts.append("▶ 취침 시간에 임박한 카페인이나 알코올 섭취는 깊은 수면을 방해하고 얕은 수면 상태에 오래 머무르게 합니다.")
            if detailed_scores.get('2.1.1_Bedtime_StdDev', 6) <= 2 or detailed_scores.get('2.1.2_Wakeup_StdDev', 6) <= 2: causal_feedback_parts.append("▶ 불규칙한 수면 일정은 멜라토닌 분비 리듬을 뒤흔들어 뇌가 깊은 수면으로 넘어가지 못하고 얕은 수면 단계에 오래 머물도록 만듭니다. 수면 시간을 최대한 규칙적으로 유지해주세요.")
        elif light_pred_val < 50:
            feedback_text = f"5. 얕은 수면 비율: 예측된 얕은 수면 비율이 {light_pred_val:.1f}%로, 이상적인 비율(50-60%)보다 낮습니다. 이는 수면 압박이 너무 높거나 특정 수면 장애의 지표일 수 있습니다."
            if detailed_scores.get('1.1_Avg_Sleep_Time', 10) == 0: causal_feedback_parts.append("▶ 평소 수면 시간이 부족하면, 우리 몸은 깊은 수면을 우선적으로 보충하려고 하여 상대적으로 얕은 수면 시간이 줄어들 수 있습니다. 하루 7시간 이상의 수면을 확보하는 것이 좋습니다.")
            if detailed_scores.get('3.4_Exercise', 0) in [1, 2, 4]: causal_feedback_parts.append("▶ 취침 2-3시간 이내에 중·고강도 운동을 하는 경우, 깊은 수면이 과도하게 늘어나면서 얕은 수면의 비율이 감소할 수 있습니다. 운동은 좋지만, 잠들기 최소 3시간 전에는 마쳐주세요.")
        if causal_feedback_parts:
            analysis_text = "\n   [원인 분석]: 아래와 같은 생활 습관의 영향일 수 있습니다.\n" + "\n".join([f"   {part}" for part in causal_feedback_parts])
    score_text = f"[점수] : {score_s1_3_3}/5"
    feedback_list.append(f"{feedback_text}{analysis_text}\n{score_text}")

    # 1.3.4 REM 수면 비율
    score_s1_3_4 = detailed_scores.get('1.3.4_REM_Sleep_Ratio', 0)
    rem_pred_val = model_predictions_for_feedback.get('REM sleep percentage', 0)
    feedback_text = ""
    analysis_text = ""
    if score_s1_3_4 == 5:
        feedback_text = f"6. REM 수면 비율: 좋습니다! 예측된 REM 수면 비율({rem_pred_val:.1f}%)이 정신 회복과 기억력 강화에 이상적인 범위입니다."
    else:
        feedback_text = f"6. REM 수면 비율: 예측된 REM 수면 비율은 {rem_pred_val:.1f}%로, 이상적인 비율(20-25%)과 차이가 있습니다. REM 수면은 학습, 기억력, 감정 조절에 매우 중요합니다."
        causal_feedback_parts = []
        if detailed_scores.get('3.2_Alcohol', 5) <= 1: causal_feedback_parts.append("▶ 알코올은 수면 후반부에 신경전달물질 불균형을 일으켜 REM 단계 진입을 강력하게 억제합니다. 이로 인해 전체 REM 수면 비율이 크게 감소할 수 있습니다.")
        causal_feedback_parts.append("▶ 이외에도 과도한 스트레스, 불안, 우울감, 또는 잠들기 직전 스마트폰의 블루라이트 노출 등도 REM 수면을 방해하는 요인이 될 수 있습니다.")
        if causal_feedback_parts:
            analysis_text = "\n   [원인 분석]: 아래와 같은 요인들이 REM 수면을 방해했을 수 있습니다.\n" + "\n".join([f"   {part}" for part in causal_feedback_parts])
    score_text = f"[점수] : {score_s1_3_4}/5"
    feedback_list.append(f"{feedback_text}{analysis_text}\n{score_text}")

    # =================================================================================
    # 2. 수면 시간 및 규칙성 피드백
    # =================================================================================
    score_s2_1_1 = detailed_scores.get('2.1.1_Bedtime_StdDev',0)
    bed_std_val = pd.to_numeric(user_input_for_feedback.get('Bedtime_std_dev'), errors='coerce')
    if pd.notna(bed_std_val):
        feedback_text = f"7. 취침 시간 규칙성: 표준편차 {bed_std_val:.1f}분으로, "
        if score_s2_1_1 == 6: feedback_text += "매우 일관적이어서 생체리듬이 안정되어 있습니다. 현재 루틴을 유지하세요."
        elif score_s2_1_1 == 4: feedback_text += "적당히 규칙적인 편이지만, 30분 이내로 더 조정해 보세요."
        elif score_s2_1_1 == 2: feedback_text += "자주 들쑥날쑥합니다. 매일 30분 이내로 맞추도록 노력해 보세요."
        else: feedback_text += "매우 불규칙합니다. 일정한 취침시간을 정해 1시간 이내 변동으로 줄이는 것이 필요합니다."
        score_text = f"[점수] : {score_s2_1_1}/6"
        feedback_list.append(f"{feedback_text}\n{score_text}")

    score_s2_1_2 = detailed_scores.get('2.1.2_Wakeup_StdDev',0)
    wake_std_val = pd.to_numeric(user_input_for_feedback.get('Wakeup_time_std_dev'), errors='coerce')
    if pd.notna(wake_std_val):
        feedback_text = f"8. 기상 시간 규칙성: 표준편차 {wake_std_val:.1f}분으로, "
        if score_s2_1_2 == 6: feedback_text += "기상 시간이 매우 일관적이어서 생체리듬이 안정됩니다. 이 루틴을 계속 유지하세요."
        elif score_s2_1_2 == 4: feedback_text += "비교적 규칙적이지만, 30분 이내로 더 맞추도록 해보세요."
        elif score_s2_1_2 == 2: feedback_text += "기상 시간이 자주 달라집니다. 매일 1시간 이내로 맞추는 것을 목표로 해보세요."
        else: feedback_text += "매우 불규칙해 생체리듬이 흔들립니다. 기상 시간을 일정하게 정해 1시간 이내로 줄여야 합니다."
        score_text = f"[점수] : {score_s2_1_2}/6"
        feedback_list.append(f"{feedback_text}\n{score_text}")

    score_s2_1_3 = detailed_scores.get('2.1.3_Weekday_Weekend_Diff',0)
    wk_wknd_diff_val = pd.to_numeric(user_input_for_feedback.get('Weekday_weekend_sleep_diff'), errors='coerce')
    if pd.notna(wk_wknd_diff_val):
        feedback_text = f"9. 주중-주말 수면 시간 차이: {wk_wknd_diff_val:.1f}시간으로, "
        if score_s2_1_3 == 5: feedback_text += "거의 일치해 생체리듬이 안정적입니다. 현재 방식을 유지하세요."
        elif score_s2_1_3 == 3: feedback_text += "약간 차이가 있지만 큰 문제는 아닙니다. 주말에도 1시간 이내로 유지해 보세요."
        elif score_s2_1_3 == 1: feedback_text += "수면 시간이 꽤 달라집니다. 주말에도 최대 1시간 차이로 맞추는 것이 좋습니다."
        else: feedback_text += "수면 패턴이 크게 흔들리고 있습니다. 주말에도 최대 1시간 이내 차이를 목표로 조정하세요."
        score_text = f"[점수] : {score_s2_1_3}/5"
        feedback_list.append(f"{feedback_text}\n{score_text}")

    score_s2_2_1 = detailed_scores.get('2.2.1_Bedtime_Consistency',0)
    bed_consistency_days_val = user_input_for_feedback.get('Bedtime_consistency_days')
    consistency_desc_map = {7:"7일 모두", 5.5:"5~6일", 3.5:"3~4일", 1:"3일 미만"}
    consistency_desc = consistency_desc_map.get(pd.to_numeric(bed_consistency_days_val, errors='coerce'), f"{bed_consistency_days_val}일")
    feedback_text = ""
    if score_s2_2_1 < 5:
        feedback_text = f"10. 취침 시간대 일관성: 주 {consistency_desc} 특정 시간대에 취침하고 계십니다. 매일 비슷한 시간대에 잠드는 것이 좋습니다."
    else:
        feedback_text = f"10. 취침 시간대 일관성: 훌륭합니다! 주 {consistency_desc} 특정 시간대에 취침하여 매우 일관적입니다."
    score_text = f"[점수] : {score_s2_2_1}/5"
    feedback_list.append(f"{feedback_text}\n{score_text}")

    score_s2_2_2 = detailed_scores.get('2.2.2_Bedtime_Slot',0)
    bed_slot_val = user_input_for_feedback.get('Bedtime_slot')
    feedback_text = ""
    if score_s2_2_2 < 5:
        feedback_text = f"11. 취침 시간대: 주로 '{bed_slot_val}'에 취침하십니다. 가급적 새벽 1시 이전에 취침하여 충분한 수면 시간을 확보하세요."
    else:
        feedback_text = f"11. 취침 시간대: 좋습니다! '{bed_slot_val}'에 취침하여 이상적인 시간대에 잠자리에 드십니다."
    score_text = f"[점수] : {score_s2_2_2}/5"
    feedback_list.append(f"{feedback_text}\n{score_text}")

    # =================================================================================
    # 3. 생활 습관 요인 피드백
    # =================================================================================
    score_s3_1 = detailed_scores.get('3.1_Caffeine',0)
    caffeine_mg_val = pd.to_numeric(user_input_for_feedback.get('Caffeine consumption'), errors='coerce')
    caffeine_last_h_val = pd.to_numeric(user_input_for_feedback.get('Caffeine_last_consumed_hours_before_bed'), errors='coerce')
    caffeine_info = ""
    if pd.notna(caffeine_mg_val) and caffeine_mg_val > 0:
        caffeine_info += f"일일 총량 {caffeine_mg_val}mg"
        if pd.notna(caffeine_last_h_val): caffeine_info += f", 마지막 섭취는 취침 {caffeine_last_h_val}시간 전"

    feedback_text = ""
    if score_s3_1 < 5 and caffeine_info:
        feedback_text = f"12. 카페인 섭취({caffeine_info}): 수면에 영향을 줄 수 있습니다. 특히 오후나 저녁에는 섭취를 줄이거나 피하세요."
    else:
        feedback_text = "12. 카페인 섭취: 카페인을 섭취하지 않거나 잘 관리하여 수면에 긍정적입니다."
    score_text = f"[점수] : {score_s3_1}/5"
    feedback_list.append(f"{feedback_text}\n{score_text}")

    score_s3_2 = detailed_scores.get('3.2_Alcohol',0)
    alcohol_ml_val = pd.to_numeric(user_input_for_feedback.get('Alcohol consumption'), errors='coerce')
    alcohol_last_h_val = pd.to_numeric(user_input_for_feedback.get('Alcohol_last_consumed_hours_before_bed'), errors='coerce')
    alcohol_info = ""
    if pd.notna(alcohol_ml_val) and alcohol_ml_val > 0:
        alcohol_info += f"섭취량 약 {alcohol_ml_val}mL"
        if pd.notna(alcohol_last_h_val): alcohol_info += f", 마지막 섭취는 취침 {alcohol_last_h_val}시간 전"

    feedback_text = ""
    if score_s3_2 < 5 and alcohol_info:
        feedback_text = f"13. 알코올 섭취({alcohol_info}): 수면의 질을 저해할 수 있습니다. 특히 잠들기 직전 음주는 피하세요."
    else:
        feedback_text = "13. 알코올 섭취: 알코올을 섭취하지 않아 수면에 매우 긍정적입니다."
    score_text = f"[점수] : {score_s3_2}/5"
    feedback_list.append(f"{feedback_text}\n{score_text}")

    score_s3_3 = detailed_scores.get('3.3_Smoking',0)
    smoking_status_val = user_input_for_feedback.get('Smoking_history_status', "정보 없음")
    feedback_text = ""
    if score_s3_3 < 5:
        feedback_text = f"14. 흡연 상태: '{smoking_status_val}' 상태는 수면에 부정적 영향을 줄 수 있습니다. 금연은 수면의 질 개선에 큰 도움이 됩니다."
    else:
        feedback_text = f"14. 흡연 상태: 비흡연 상태('{smoking_status_val}')를 유지하여 수면에 긍정적입니다."
    score_text = f"[점수] : {score_s3_3}/5"
    feedback_list.append(f"{feedback_text}\n{score_text}")

    score_s3_4 = detailed_scores.get('3.4_Exercise',0)
    mod_mins_val = user_input_for_feedback.get('Exercise_aerobic_moderate_minutes',0)
    vig_mins_val = user_input_for_feedback.get('Exercise_aerobic_vigorous_minutes',0)
    ex_last_h_str_val = user_input_for_feedback.get('Exercise_last_hours_before_bed', "해당 없음")
    ex_timing_fb = ""
    if ex_last_h_str_val != "해당 없음": ex_timing_fb = f"(최근 운동은 취침 {ex_last_h_str_val}시간 전)"

    feedback_text = ""
    if pd.to_numeric(mod_mins_val,errors='coerce') == 0 and pd.to_numeric(vig_mins_val,errors='coerce') == 0:
        feedback_text = "15. 운동 습관: 규칙적인 신체 활동은 수면의 질을 크게 향상시킵니다. 가벼운 운동부터 시작해보세요."
    elif score_s3_4 < 5:
        feedback_text = f"15. 운동 습관: 현재 운동량(중강도 {mod_mins_val}분, 고강도 {vig_mins_val}분)이나 시간이 최적이 아닐 수 있습니다. {ex_timing_fb} 규칙적인 운동은 좋지만, 잠들기 최소 3시간 전에는 마치고, 주당 권장량을 채워보세요."
    else:
        feedback_text = f"15. 운동 습관: 훌륭합니다! 운동량(중강도 {mod_mins_val}분, 고강도 {vig_mins_val}분)과 시간을 잘 지켜 수면의 질을 높이고 계십니다. {ex_timing_fb}"
    score_text = f"[점수] : {score_s3_4}/5"
    feedback_list.append(f"{feedback_text}\n{score_text}")

    return feedback_list

In [ ]:
# ===========================================================================
# 8-1. 요약 피드백 생성 함수 (새로 추가)
# ===========================================================================
def generate_summary_feedback(total_score, detailed_scores, quality_evaluation):
    """
    점수표와 시나리오를 바탕으로 요약 피드백 문장을 생성합니다.
    (수면점수에 대한 피드백.md 문서 기반)
    """
    # 점수가 낮은 항목을 찾기 위한 기준 (최대 점수의 50% 미만)
    low_score_thresholds = {
        '1.1_Avg_Sleep_Time': 5, '1.2_Awakenings_Count': 5, '1.3.1_Sleep_Efficiency': 7,
        '1.3.2_Deep_Sleep_Ratio': 5, '1.3.3_Light_Sleep_Ratio': 2.5, '1.3.4_REM_Sleep_Ratio': 2.5,
        '2.1.1_Bedtime_StdDev': 3, '2.1.2_Wakeup_StdDev': 3, '2.1.3_Weekday_Weekend_Diff': 2.5,
        '2.2.1_Bedtime_Consistency': 2.5, '2.2.2_Bedtime_Slot': 2.5,
        '3.1_Caffeine': 2.5, '3.2_Alcohol': 2.5, '3.3_Smoking': 2.5, '3.4_Exercise': 2.5
    }
    # 사용자에게 보여줄 항목 이름
    category_names = {
        '1.1_Avg_Sleep_Time': '평균 수면 시간', '1.2_Awakenings_Count': '수면 중 각성 횟수',
        '1.3.1_Sleep_Efficiency': '수면 효율성', '1.3.2_Deep_Sleep_Ratio': '깊은 수면 비율',
        '1.3.3_Light_Sleep_Ratio': '얕은 수면 비율', '1.3.4_REM_Sleep_Ratio': 'REM 수면 비율',
        '2.1.1_Bedtime_StdDev': '취침 시간 규칙성', '2.1.2_Wakeup_StdDev': '기상 시간 규칙성',
        '2.1.3_Weekday_Weekend_Diff': '주중/주말 수면 시간 차이', '2.2.1_Bedtime_Consistency': '취침 시간대 일관성',
        '2.2.2_Bedtime_Slot': '취침 시간대', '3.1_Caffeine': '카페인 섭취 습관', '3.2_Alcohol': '알코올 섭취 습관',
        '3.3_Smoking': '흡연 습관', '3.4_Exercise': '운동 습관'
    }

    # 1. 개선이 필요한 항목 목록 생성
    problematic_items = [
        name for key, name in category_names.items()
        if key in detailed_scores and detailed_scores[key] < low_score_thresholds.get(key, 0)
    ]

    # 2. 수면 비율(깊은/얕은/REM) 상태 확인
    deep_sleep_ok = detailed_scores.get('1.3.2_Deep_Sleep_Ratio', 0) >= 8 # 8점 이상이면 양호
    light_sleep_ok = detailed_scores.get('1.3.3_Light_Sleep_Ratio', 0) == 5 # 5점 만점이어야 양호
    rem_sleep_ok = detailed_scores.get('1.3.4_REM_Sleep_Ratio', 0) == 5 # 5점 만점이어야 양호
    ratios_are_good = deep_sleep_ok and light_sleep_ok and rem_sleep_ok

    # 3. 피드백 문장 조합
    base_sentence = f"현재 사용자의 경우, 수면 점수는 {total_score}점으로 '{quality_evaluation}'에 해당합니다."
    overall_assessment = ""
    improvement_suggestion = ""

    # 시나리오 1: 수면 점수가 높음 (80점 이상)
    if total_score >= 80:
        if not problematic_items:
            overall_assessment = "대부분의 수면 요소가 양호하며 매우 건강한 상태로 보입니다."
        else: # 점수는 높지만 일부 낮은 항목이 있는 경우
            overall_assessment = "대부분의 수면 요소가 양호하신 상태입니다."
            improvement_suggestion = f"하지만 '{', '.join(problematic_items)}' 부분에서 개선의 여지가 있으니, 관련 피드백을 참고하여 보완하시면 더욱 좋겠습니다."

    # 시나리오 2 & 3: 수면 점수가 낮음 (80점 미만)
    else:
        if ratios_are_good: # 시나리오 2: 점수는 낮지만 수면 비율은 적절
            overall_assessment = "수면 단계 비율은 비교적 괜찮지만, 다른 습관들이 수면의 질을 떨어뜨리고 있습니다."
            if problematic_items:
                improvement_suggestion = f"특히 '{', '.join(problematic_items)}' 항목들의 점수가 낮으니, 이 부분들에 대한 집중적인 개선이 필요합니다."
        else: # 시나리오 3: 점수도 낮고 수면 비율도 부적절
            problem_ratios = []
            if not deep_sleep_ok: problem_ratios.append('깊은 수면')
            if not light_sleep_ok: problem_ratios.append('얕은 수면')
            if not rem_sleep_ok: problem_ratios.append('REM 수면')

            overall_assessment = f"특히 '{', '.join(problem_ratios)}' 비율이 적절치 못해 수면의 질이 크게 낮아진 것으로 보입니다."
            related_problems = []
            if not deep_sleep_ok and '카페인 섭취 습관' in problematic_items: related_problems.append('카페인 섭취')
            if not deep_sleep_ok and '알코올 섭취 습관' in problematic_items: related_problems.append('알코올 섭취')
            if not light_sleep_ok and '취침 시간 규칙성' in problematic_items: related_problems.append('불규칙한 수면 시간')

            if related_problems:
                improvement_suggestion = f"이는 '{', '.join(list(set(related_problems)))}'과 같은 생활 습관과 연관되었을 가능성이 높습니다. 관련 항목들의 개선이 시급합니다."
            elif problematic_items:
                 improvement_suggestion = f"'{', '.join(problematic_items)}'와 같은 다른 습관들도 함께 개선해야 합니다."

    return f"{base_sentence} {overall_assessment} {improvement_suggestion}".strip()

In [ ]:
# ===========================================================================
# 9. CLI 입력 및 유효성 검사 함수 (survey_prompter.py 기반)
# ===========================================================================
def _is_float(value):
    """문자열이 부동소수점으로 변환 가능한지 확인합니다."""
    try:
        n = float(value)
        return True # NaN, Inf도 float으로 간주 (추후 처리)
    except ValueError:
        return False

def _is_int(value, r_min=-float('inf'), r_max=float('inf')):
    """문자열이 정수로 변환 가능하고 지정된 범위 내에 있는지 확인합니다."""
    try:
        n = int(value)
        return r_min <= n <= r_max
    except ValueError:
        return False

def _is_alphabet(value, domain):
    """문자열이 지정된 도메인 내의 단일 알파벳인지 확인합니다 (대소문자 구분 없음)."""
    ch = value.strip().upper()
    return len(ch) == 1 and ch in domain

def calculate_pack_years(cigarettes_per_day_str, smoking_years_str):
    """갑년(pack-years)을 계산합니다."""
    try:
        cigarettes_per_day = int(cigarettes_per_day_str)
        smoking_years = int(smoking_years_str)
        if cigarettes_per_day < 0 or smoking_years < 0: return None # 음수 입력 방지
        return (cigarettes_per_day / 20) * smoking_years
    except ValueError:
        return None # 숫자 변환 실패 시

def run_survey_for_notebook():
    """CLI를 통해 사용자 입력을 받고, 노트북 추론 파이프라인에 맞는 딕셔너리로 반환합니다."""
    questions = OrderedDict([
        ('user_status', OrderedDict([
            ('Age', ('나이가 어떻게 되시나요?', '숫자 (18-65)', lambda x: _is_int(x, 18, 65))),
            ('Gender', ('성별이 어떻게 되시나요?', 'M(남성) / F(여성)', lambda x: _is_alphabet(x, ['M', 'F']))),
        ])),
        ('sleeping_pattern', OrderedDict([
            ('Sleep duration', ('평균적으로, 하루 몇 시간 주무시나요?', '숫자 (예: 7.5)', lambda x: _is_float(x) and 0 <= float(x) <= 24)),
            ('Awakenings', ('평균적으로, 하룻밤 동안 깨는 횟수가 어떻게 되시나요? (모르면 "잘 모르겠다" 입력)', '숫자 (0-10) 또는 "잘 모르겠다"',
                           lambda x: x.lower() == "잘 모르겠다" or (_is_int(x) and 0 <= int(x) <= 10))),
            ('Bedtime_day1', ('일주일동안의 취침 시간을 입력받습니다. Day 1 취침 시각을 입력해주세요.', ' (분 단위, 예: 오후 11:30이면 690, 새벽 1:00이면 780)', lambda x: _is_int(x, 0) and int(x) <= 24*60)),
            ('Bedtime_day2', ('Day 2 취침 시각을 입력해주세요.', '(분 단위, 예: 오후 11:30이면 690, 새벽 1:00이면 780)', lambda x: _is_int(x, 0) and int(x) <= 24*60)),
            ('Bedtime_day3', ('Day 3 취침 시각을 입력해주세요.', ' (분 단위, 예: 오후 11:30이면 690, 새벽 1:00이면 780)', lambda x: _is_int(x, 0) and int(x) <= 24*60)),
            ('Bedtime_day4', ('Day 4 취침 시각을 입력해주세요.', ' (분 단위, 예: 오후 11:30이면 690, 새벽 1:00이면 780)', lambda x: _is_int(x, 0) and int(x) <= 24*60)),
            ('Bedtime_day5', ('Day 5 취침 시각을 입력해주세요.', ' (분 단위, 예: 오후 11:30이면 690, 새벽 1:00이면 780)', lambda x: _is_int(x, 0) and int(x) <= 24*60)),
            ('Bedtime_day6', ('Day 6 취침 시각을 입력해주세요.', ' (분 단위, 예: 오후 11:30이면 690, 새벽 1:00이면 780)', lambda x: _is_int(x, 0) and int(x) <= 24*60)),
            ('Bedtime_day7', ('Day 7 취침 시각을 입력해주세요.', ' (분 단위, 예: 오후 11:30이면 690, 새벽 1:00이면 780)', lambda x: _is_int(x, 0) and int(x) <= 24*60)),
            ('Wakeup_day1', ('일주일동안의 기상 시간을 입력받습니다. Day 1 기상 시각을 입력해주세요.', ' (분 단위, 예: 오전 7:40분이면 460, 오전 9:00이면 540)', lambda x: _is_int(x, 0) and int(x) <= 24*60)),
            ('Wakeup_day2', ('Day 2 기상 시각을 입력해주세요.', ' (분 단위, 예: 오전 7:40분이면 460, 오전 9:00이면 540)', lambda x: _is_int(x, 0) and int(x) <= 24*60)),
            ('Wakeup_day3', ('Day 3 기상 시각을 입력해주세요.', ' (분 단위, 예: 오전 7:40분이면 460, 오전 9:00이면 540)', lambda x: _is_int(x, 0) and int(x) <= 24*60)),
            ('Wakeup_day4', ('Day 4 기상 시각을 입력해주세요.', ' (분 단위, 예: 오전 7:40분이면 460, 오전 9:00이면 540)', lambda x: _is_int(x, 0) and int(x) <= 24*60)),
            ('Wakeup_day5', ('Day 5 기상 시각을 입력해주세요.', ' (분 단위, 예: 오전 7:40분이면 460, 오전 9:00이면 540)', lambda x: _is_int(x, 0) and int(x) <= 24*60)),
            ('Wakeup_day6', ('Day 6 기상 시각을 입력해주세요.', ' (분 단위, 예: 오전 7:40분이면 460, 오전 9:00이면 540)', lambda x: _is_int(x, 0) and int(x) <= 24*60)),
            ('Wakeup_day7', ('Day 7 기상 시각을 입력해주세요.', ' (분 단위, 예: 오전 7:40분이면 460, 오전 9:00이면 540)', lambda x: _is_int(x, 0) and int(x) <= 24*60)),
            ('Weekday_weekend_sleep_diff', ('주중과 주말의 평균 수면 시간 차이는 어느 정도 되시나요? (시간 단위, 주말이 더 길면 양수)', '숫자 (예: 1.5 또는 -0.5)', lambda x: _is_float(x))),
            ('Bedtime_consistency_days', ('최근 일주일 중, 평소 취침 시간대(2시간 범위 내)에 잠든 날은 며칠인가요?', '숫자 (0-7)', lambda x: _is_int(x, 0, 7))),
            ('Bedtime_slot', ('평균적인 취침 시간대는 어떻게 되시나요?\n 1. 새벽 1시 이전\n 2. 새벽 1시-2시 사이\n 3. 새벽 2시-3시 사이\n 4. 새벽 3시 이후', '숫자 (1-4)', lambda x: _is_int(x, 1, 4))),
        ])),
        ('lifestyle_habits', OrderedDict([
            ('Caffeine consumption', ('하루 평균 카페인 섭취량은 어느 정도인가요? (mg 단위)', '숫자 (예: 100)', lambda x: _is_float(x) and float(x) >=0)),
            ('Caffeine_last_consumed_hours_before_bed', ('마지막 카페인 섭취 후 몇 시간 뒤에 주무시나요? (섭취 안하면 0)', '숫자 (예: 8)', lambda x: _is_float(x) and float(x) >=0)),
            ('Alcohol consumption', ('하루 평균 알코올 섭취량은 어느 정도인가요? (mL 단위, 순수 알코올 아님)', '숫자 (예: 350)', lambda x: _is_float(x) and float(x) >=0)),
            ('Alcohol_last_consumed_hours_before_bed', ('마지막 알코올 섭취 후 몇 시간 뒤에 주무시나요? (섭취 안하면 0)', '숫자 (예: 3)', lambda x: _is_float(x) and float(x) >=0)),
            ('Smoking status_model', ('현재 담배를 피우시나요? (모델 학습용)', 'Y / N', lambda x: _is_alphabet(x, ['Y', 'N']))),
            ('Exercise frequency', ('일주일에 몇 번 정도 운동하시나요? (모델 입력용, 0-7회)', '숫자 (0-7)', lambda x: _is_int(x, 0, 7))), # Exercise frequency 직접 입력
            ('Exercise_aerobic_moderate_minutes', ('일주일에 중강도 유산소 운동을 몇 분 하시나요? (점수 계산용)', '숫자 (예: 150)', lambda x: _is_int(x, 0))),
            ('Exercise_aerobic_vigorous_minutes', ('일주일에 고강도 유산소 운동을 몇 분 하시나요? (점수 계산용)', '숫자 (예: 75)', lambda x: _is_int(x, 0))),
            ('Exercise_last_hours_before_bed', ('최근 운동을 마친 시간으로부터 실제 잠들기까지 몇 시간의 간격이 있었나요? (운동 안했거나 해당 없으면 "해당 없음" 입력)', '숫자 (예: 3) 또는 "해당 없음"',
                                              lambda x: x.lower() == "해당 없음" or (_is_float(x) and float(x) >= 0) ))
        ]))
    ])

    survey_answers_raw = OrderedDict()
    print("수면 습관 개선을 위한 설문을 시작합니다. 각 질문에 답변해주세요.")
    for section, components in questions.items():
        survey_answers_raw[section] = OrderedDict()
        print(f"\n--- {section.replace('_', ' ').title()} ---")
        for keyword, (prompt, constraints, validator) in components.items():
            while True:
                answer = input(f"{prompt} ({constraints}): ")
                if validator(answer):
                    break
                print("입력이 올바르지 않습니다. 다시 입력해 주세요.")
            survey_answers_raw[section][keyword] = answer.strip()

    bedtimes = [int(survey_answers_raw['sleeping_pattern'][f'Bedtime_day{i}']) for i in range(1, 8)]
    bedtimes_array = np.array(bedtimes, dtype=float)
    bedtime_std = float(np.std(bedtimes_array, ddof=0))
    survey_answers_raw['sleeping_pattern']['Bedtime_std_dev'] = f"{bedtime_std:.2f}"

    wakeups = [int(survey_answers_raw['sleeping_pattern'][f'Wakeup_day{i}']) for i in range(1, 8)]
    wakeups_array = np.array(wakeups, dtype=float)
    wakeup_std = float(np.std(wakeups_array, ddof=0))
    survey_answers_raw['sleeping_pattern']['Wakeup_time_std_dev'] = f"{wakeup_std:.2f}"

    smoking_now = survey_answers_raw['lifestyle_habits']['Smoking status_model'].upper() == 'Y'
    smoking_history_status_final = "비흡연자"
    if smoking_now:
        while True:
            cigs_day = input("하루 평균 몇 개비 피우시나요? (숫자): ")
            smoke_years = input("흡연 기간은 총 몇 년인가요? (숫자): ")
            pack_years = calculate_pack_years(cigs_day, smoke_years)
            if pack_years is not None:
                smoking_history_status_final = "현재흡연_중증" if pack_years >= 10 else "현재흡연_경증"
                break
            print("갑년 계산에 필요한 입력이 올바르지 않습니다. 다시 입력해주세요.")
    else:
        past_smoker = input("과거에 담배를 피운 경험이 있으신가요? (Y/N): ").strip().upper()
        if past_smoker == 'Y':
            while True:
                quit_years = input("금연한 지 몇 년 되셨나요? (숫자, 1년 미만은 0): ")
                if _is_int(quit_years, 0):
                    smoking_history_status_final = "과거흡연_1년이상" if int(quit_years) >= 1 else "과거흡연_1년미만"
                    break
                print("금연 기간 입력이 올바르지 않습니다.")
    survey_answers_raw['lifestyle_habits']['Smoking_history_status_for_score'] = smoking_history_status_final

    mapped_input = {}
    mapped_input['Age'] = survey_answers_raw['user_status']['Age']
    mapped_input['Gender'] = 'Male' if survey_answers_raw['user_status']['Gender'].upper() == 'M' else 'Female'
    mapped_input['Sleep duration'] = survey_answers_raw['sleeping_pattern']['Sleep duration']
    mapped_input['Awakenings'] = survey_answers_raw['sleeping_pattern']['Awakenings']
    mapped_input['Bedtime_std_dev'] = survey_answers_raw['sleeping_pattern']['Bedtime_std_dev']
    mapped_input['Wakeup_time_std_dev'] = survey_answers_raw['sleeping_pattern']['Wakeup_time_std_dev']
    mapped_input['Weekday_weekend_sleep_diff'] = survey_answers_raw['sleeping_pattern']['Weekday_weekend_sleep_diff']
    mapped_input['Bedtime_consistency_days'] = survey_answers_raw['sleeping_pattern']['Bedtime_consistency_days']
    bedtime_choice = int(survey_answers_raw['sleeping_pattern']['Bedtime_slot'])
    bedtime_map = {1: '새벽 1시 이전 취침', 2: '새벽 1-2시 사이 취침', 3: '새벽 2-3시 사이 취침', 4: '새벽 3시 이후 취침'}
    mapped_input['Bedtime_slot'] = bedtime_map.get(bedtime_choice)
    mapped_input['Caffeine consumption'] = survey_answers_raw['lifestyle_habits']['Caffeine consumption']
    mapped_input['Caffeine_last_consumed_hours_before_bed'] = survey_answers_raw['lifestyle_habits']['Caffeine_last_consumed_hours_before_bed']
    mapped_input['Alcohol consumption'] = survey_answers_raw['lifestyle_habits']['Alcohol consumption']
    mapped_input['Alcohol_last_consumed_hours_before_bed'] = survey_answers_raw['lifestyle_habits']['Alcohol_last_consumed_hours_before_bed']
    mapped_input['Smoking status'] = 'Yes' if survey_answers_raw['lifestyle_habits']['Smoking status_model'].upper() == 'Y' else 'No'
    mapped_input['Smoking_history_status'] = survey_answers_raw['lifestyle_habits']['Smoking_history_status_for_score']

    # Exercise frequency를 직접 받은 값으로 사용 (모델 입력용)
    mapped_input['Exercise frequency'] = survey_answers_raw['lifestyle_habits']['Exercise frequency']

    # 점수 계산용 상세 운동 정보
    mapped_input['Exercise_aerobic_moderate_minutes'] = survey_answers_raw['lifestyle_habits']['Exercise_aerobic_moderate_minutes']
    mapped_input['Exercise_aerobic_vigorous_minutes'] = survey_answers_raw['lifestyle_habits']['Exercise_aerobic_vigorous_minutes']
    mapped_input['Exercise_last_hours_before_bed'] = survey_answers_raw['lifestyle_habits']['Exercise_last_hours_before_bed']

    print("\n--- Mapped User Input for Notebook ---")
    for k, v in mapped_input.items(): print(f"'{k}': '{v}' (type: {type(v)})")
    return mapped_input

In [ ]:
# ===========================================================================
# 10. 메인 실행 로직 (CLI 입력 통합 및 학습/추론 실행)
# ===========================================================================
if __name__ == "__main__":
    # 1. Kaggle API 설정 및 데이터셋 로드
    setup_kaggle_api(KAGGLE_CONFIG_PATH_DRIVE)
    raw_df_loaded = download_and_load_dataset('equilibriumm/sleep-efficiency', DATASET_DOWNLOAD_PATH, 'Sleep_Efficiency.csv')

    # ##############################################################################
    # ★★★ 데이터셋 컬럼 스왑(Swap) 적용 ★★★
    # 'Deep sleep percentage'와 'Light sleep percentage'의 데이터가 바뀐 문제를 해결하기 위해
    # 데이터프레임을 로드한 직후 컬럼 이름을 서로 교체합니다.
    # ##############################################################################
    if not raw_df_loaded.empty:
        print("\nSwapping 'Deep sleep percentage' and 'Light sleep percentage' columns to correct dataset issue...")
        raw_df_loaded.rename(columns={
            'Deep sleep percentage': 'Light sleep percentage',
            'Light sleep percentage': 'Deep sleep percentage'
        }, inplace=True)
        print("Column swap complete.\n")


    # 2. 타겟별 모델 및 하이퍼파라미터 설정
    # 컬럼 데이터 스왑에 맞춰 모델 설정도 명시적으로 스왑합니다.
    TARGET_SPECIFIC_MODEL_CONFIGS = {
        'Sleep efficiency': {'model_name': 'XGBoost', 'param_grid': {'n_estimators': [50, 75, 100, 125, 150], 'learning_rate': [0.01, 0.025, 0.05, 0.075, 0.1], 'max_depth': [1, 2, 3, 4], 'min_child_weight': [1, 2, 3]}},
        'REM sleep percentage': {'model_name': 'RandomForest', 'param_grid': {'n_estimators': [100, 150], 'max_depth': [10, 20], 'min_samples_split': [2, 5], 'min_samples_leaf': [2, 4]}},
        # 원래 'Light sleep percentage'에 사용하려던 설정을 이제 'Deep sleep percentage'에 적용
        'Deep sleep percentage': {'model_name': 'SVR', 'param_grid': {'C': [1, 10], 'kernel': ['rbf'], 'gamma': ['scale', 'auto']}},
        # 원래 'Deep sleep percentage'에 사용하려던 설정을 이제 'Light sleep percentage'에 적용
        'Light sleep percentage': {'model_name': 'SVR', 'param_grid': {'C': [1, 10], 'kernel': ['rbf'], 'gamma': ['scale', 'auto']}}
    }

    # 3. 학습 파이프라인 실행 여부 결정
    RUN_TRAINING_PIPELINE = True # True로 설정하면 항상 학습 실행, False면 기존 모델 사용 시도
    models_exist = os.path.exists(os.path.join(MODEL_SAVE_DIR, 'scalers.pkl')) # 대표 파일로 존재 여부 확인

    if RUN_TRAINING_PIPELINE or not models_exist:
        print("\n\n========== RUNNING TRAINING PIPELINE ==========")
        if raw_df_loaded.empty:
            print("Dataset is empty. Training cannot proceed.")
        else:
            trained_models_dict, scalers_dict, medians_dict, final_features_list_from_training = run_training_pipeline(
                raw_df_loaded, BASE_SELECTED_FEATURES, TARGET_COLUMNS, TARGET_SPECIFIC_MODEL_CONFIGS, MODEL_SAVE_DIR
            )
    else:
        print("\n\n========== SKIPPING TRAINING - Attempting to use existing models. ==========")

    # 4. CLI 설문 실행 및 추론/점수계산/피드백 실행
    if os.path.exists(os.path.join(MODEL_SAVE_DIR, 'scalers.pkl')): # 학습된 모델(전처리기)이 있어야 추론 가능
        print("\n\n========== STARTING USER SURVEY AND INFERENCE PIPELINE ==========")
        user_input_data_mapped = run_survey_for_notebook() # 실제 설문 실행
        # --- 테스트용 샘플 데이터 ---
        # user_input_data_mapped = {
        #     'Age': '25', 'Gender': 'Female', 'Sleep duration': '6.5', 'Awakenings': '1',
        #     'Bedtime_std_dev': '45.3', 'Wakeup_time_std_dev': '25.8', 'Weekday_weekend_sleep_diff': '1.5',
        #     'Bedtime_consistency_days': '5.5', 'Bedtime_slot': '새벽 1-2시 사이 취침',
        #     'Caffeine consumption': '150', 'Caffeine_last_consumed_hours_before_bed': '6',
        #     'Alcohol consumption': '0', 'Alcohol_last_consumed_hours_before_bed': '0',
        #     'Smoking status': 'No', 'Smoking_history_status': '비흡연자',
        #     'Exercise frequency': '3', 'Exercise_aerobic_moderate_minutes': '90',
        #     'Exercise_aerobic_vigorous_minutes': '0', 'Exercise_last_hours_before_bed': '4'
        # }
        # print("--- Using Sample User Input for Testing ---")


        user_model_predictions, user_original_input_for_score_calc = run_inference_pipeline(
            user_input_data_mapped, TARGET_COLUMNS, TARGET_SPECIFIC_MODEL_CONFIGS, MODEL_SAVE_DIR
        )

        if user_model_predictions:
            print("\n--- Calculating Sleep Score ---")
            total_score, detailed_scores_breakdown = calculate_sleep_score(
                user_original_input_for_score_calc, user_model_predictions
            )
            quality_evaluation, general_interpretation_message = get_score_interpretation(total_score)

            print(f"\n========== 최종 수면 점수 및 분석 ({user_original_input_for_score_calc.get('Age')}세 {user_original_input_for_score_calc.get('Gender')}) ==========")

            # 1. 요약 피드백 생성 및 출력
            summary_feedback = generate_summary_feedback(total_score, detailed_scores_breakdown, quality_evaluation)
            print(">>> 요약 피드백 <<<")
            print(summary_feedback)
            print("-" * 20)

            # 2. 기존 결과 출력 (총점 및 세부 점수)
            print(f"총 수면 점수: {total_score}점 ({quality_evaluation})")
            print(f"분석: {general_interpretation_message}")

            print("\n세부 항목별 점수:")
            for category, score_val in detailed_scores_breakdown.items():
                formatted_cat_name = category.replace('_', ' ').replace('Total', ' Total').replace('Ratio', ' Ratio').replace('StdDev', ' Std Dev').replace('Diff', ' Diff')
                print(f"  - {formatted_cat_name}: {score_val}점")

            # ================== 수정된 부분 시작 ==================
            print("\n--- 세부 개선 조언 ---")
            detailed_feedback_messages = generate_detailed_feedback(detailed_scores_breakdown, user_original_input_for_score_calc, user_model_predictions)
            if detailed_feedback_messages:
                for feedback_block in detailed_feedback_messages:
                    print(feedback_block)
                    print()  # 각 피드백 묶음 사이에 빈 줄 추가
            else:
                print("모든 항목에서 훌륭한 수면 습관을 가지고 계십니다! 이대로 꾸준히 유지해주세요.")
            # ================== 수정된 부분 끝 ==================
        else:
            print("모델 예측값을 얻는 데 실패하여 수면 점수 계산 및 피드백 생성을 진행할 수 없습니다.")
    else:
        print("\n학습된 모델 또는 전처리 도구가 없습니다. 먼저 학습 파이프라인을 실행해주세요 (RUN_TRAINING_PIPELINE = True).")

Kaggle API key configured successfully.
Dataset URL: https://www.kaggle.com/datasets/equilibriumm/sleep-efficiency
Error in dataset download/load: KaggleHttpClient.call() got an unexpected keyword argument 'headers'


========== RUNNING TRAINING PIPELINE ==========
Dataset is empty. Training cannot proceed.


========== STARTING USER SURVEY AND INFERENCE PIPELINE ==========
수면 습관 개선을 위한 설문을 시작합니다. 각 질문에 답변해주세요.

--- User Status ---
나이가 어떻게 되시나요? (숫자 (18-65)): 25
성별이 어떻게 되시나요? (M(남성) / F(여성)): M

--- Sleeping Pattern ---
평균적으로, 하루 몇 시간 주무시나요? (숫자 (예: 7.5)): 5.5
평균적으로, 하룻밤 동안 깨는 횟수가 어떻게 되시나요? (모르면 "잘 모르겠다" 입력) (숫자 (0-10) 또는 "잘 모르겠다"): 3
일주일동안의 취침 시간을 입력받습니다. Day 1 취침 시각을 입력해주세요. ( (분 단위, 예: 오후 11:30이면 690, 새벽 1:00이면 780)): 810
Day 2 취침 시각을 입력해주세요. ((분 단위, 예: 오후 11:30이면 690, 새벽 1:00이면 780)): 900
Day 3 취침 시각을 입력해주세요. ( (분 단위, 예: 오후 11:30이면 690, 새벽 1:00이면 780)): 990
Day 4 취침 시각을 입력해주세요. ( (분 단위, 예: 오후 11:30이면 690, 새벽 1:00이면 780)): 840
Day 5 취침 시각을 입력해주세요. ( (분 단위, 예: 오후 11:30이면 690, 새벽 1:00